In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path(".")

files = [
    ("A_7_5g",  ROOT/"APuMn_7_5g"/"APuMn_7_5g_clean.csv"),
    ("A_15g",   ROOT/"APuMn_15g"/"APuMn_15g_clean.csv"),
    ("A_18g",   ROOT/"APuMn_18g"/"APuMn_18g_clean.csv"),
    ("A_22_5g", ROOT/"APuMn_22_5g"/"APuMn_22_5g_clean.csv"),
]

def load(path):
    df = pd.read_csv(path)
    # Arreglar por si quedaron nombres raros:
    df = df.rename(columns={c.lower().strip():c for c in df.columns})
    if not {"nm","A"}.issubset(df.columns):
        # Si venía como x,y entonces lo corregimos
        c0, c1 = df.columns[:2]
        df = df.rename(columns={c0:"nm", c1:"A"})
    df["nm"] = pd.to_numeric(df["nm"], errors="coerce")
    df["A"]  = pd.to_numeric(df["A"], errors="coerce")
    return df.dropna()[["nm","A"]].sort_values("nm").drop_duplicates()

# ---- MATRIZ SIN INTERPOLAR (INTERSECCIÓN EXACTA) ----

mat = load(files[0][1]).rename(columns={"A": files[0][0]})

for label, path in files[1:]:
    df = load(path).rename(columns={"A": label})
    mat = mat.merge(df, on="nm", how="inner")   # SOLO nm que existen en todos

mat = mat.sort_values("nm").reset_index(drop=True)

out = ROOT / "APuMn_matrix.csv"
mat.to_csv(out, index=False)

print("✔ Matriz generada correctamente:")
print(out)
display(mat.head())

✔ Matriz generada correctamente:
APuMn_matrix.csv


,nm,A_7_5g,A_15g,A_18g,A_22_5g
0,230.5,1.051,2.449,1.234,1.654
1,231.0,1.040,2.392,1.220,1.635
2,231.5,1.030,2.337,1.207,1.616
3,232.0,1.020,2.290,1.195,1.597
4,232.5,1.009,2.249,1.182,1.578
